In [1]:
import os
import json
import requests
from tqdm import tqdm
from typing import List, Optional

import fitz
import torch
import numpy as np
from google import genai
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from pymilvus import MilvusClient, FieldSchema, DataType, CollectionSchema

from sqlalchemy.engine import URL
from pgvector.sqlalchemy import Vector
from sqlalchemy.orm import DeclarativeBase, Mapped, Session, mapped_column
from sqlalchemy import Integer, String, Float, Boolean, select, create_engine

In [2]:
db_url = URL.create(
    drivername="postgresql+psycopg",
    username="postgres",
    password="password",
    host="localhost",
    port=5432,
    database="similarity_search_service_db"
)

In [3]:
# Create the base class for the table definition
class Base(DeclarativeBase):
    __abstract__ = True


# Create the table definition
class Images(Base):
    __tablename__ = "images"
    VECTOR_LENGTH = 512
    
    # primary key
    id: Mapped[int] = mapped_column(Integer, primary_key=True)
    # image path - we will use it to store the path to the image file, 
    # after similarity search we can use it to retrieve the image and display it
    image_path: Mapped[str] = mapped_column(String(256))
    # image embedding - we will store the image embedding in this column, 
    # the image embedding is a list of 512 floats this is the 
    # output of the sentence transformer model
    image_embedding: Mapped[List[float]] = mapped_column(Vector(VECTOR_LENGTH))

In [4]:
# Create SQLAlchemy engine and initialize tables
engine = create_engine(db_url)
Base.metadata.create_all(engine)

In [5]:
def insert_image(engine, image_path: str, image_embedding: list[float]):
    with Session(engine) as session:
        # create the image object
        image = Images(image_path=image_path, image_embedding=image_embedding)
        # add the image object to the session
        session.add(image)
        # commit the transaction
        session.commit()

# insert some data into the table
N = 100
for i in range(N):
    image_path = f"image_{i}.jpg"
    image_embedding = np.random.rand(512).tolist()
    insert_image(engine, image_path, image_embedding)

# select first image from the table
with Session(engine) as session:
    image = session.query(Images).first()

def find_k_images(engine, k: int, original_image: Images):
    with Session(engine) as session:
        result = session.execute(
            select(Images)
            .order_by(Images.image_embedding.cosine_distance(original_image.image_embedding))
            .limit(k),
            execution_options={"prebuffer_rows": True}
        )
        return list(result.scalars().all())

top10 = find_k_images(engine, 10, image) # type: ignore
[img.image_path for img in top10]

['image_0.jpg',
 'image_32.jpg',
 'image_77.jpg',
 'image_1.jpg',
 'image_26.jpg',
 'image_62.jpg',
 'image_9.jpg',
 'image_29.jpg',
 'image_84.jpg',
 'image_53.jpg']

In [7]:
def find_images_with_similarity_score_greater_than(engine, similarity_score: float, original_image: Images):
    with Session(engine) as session:
        # Since cosine_similarity = 1 - cosine_distance, filter by distance < 1 - similarity_score
        distance_threshold = 1.0 - similarity_score
        result = session.execute(
            select(Images)
            .filter(Images.image_embedding.cosine_distance(original_image.image_embedding) < distance_threshold),
            execution_options={"prebuffer_rows": True}
        )
        return list(result.scalars().all())

filtered = find_images_with_similarity_score_greater_than(engine, 0.9, image) # type: ignore
len(filtered)

1

Steam Games dataset: table, embeddings, and search

In [8]:
class Games(Base):
    __tablename__ = 'games'
    __table_args__ = {'extend_existing': True}
    VECTOR_LENGTH = 512  # distiluse-base-multilingual-cased-v2 outputs 512-d vectors
    id: Mapped[int] = mapped_column(Integer, primary_key=True)
    name: Mapped[str] = mapped_column(String(256))
    description: Mapped[str] = mapped_column(String(4096))
    windows: Mapped[bool] = mapped_column(Boolean)
    linux: Mapped[bool] = mapped_column(Boolean)
    mac: Mapped[bool] = mapped_column(Boolean)
    price: Mapped[float] = mapped_column(Float)
    game_description_embedding: Mapped[List[float]] = mapped_column(Vector(VECTOR_LENGTH))

# Recreate schema for fresh run
Base.metadata.drop_all(engine)
Base.metadata.create_all(engine)

# Load dataset subset
dataset = load_dataset('FronkonGames/steam-games-dataset')
columns = dataset['train'].features # type: ignore
print('Columns:', columns)

columns_to_keep = ['Name', 'Windows', 'Linux', 'Mac', 'About the game', 'Price']
N = 4000  # Keep it smaller for faster demo; adjust to 40000 if you have time
dataset = dataset['train'].select_columns( # type: ignore
    columns_to_keep).select(range(min(N, len(dataset['train'])))) # type: ignore

# Embedding model
checkpoint = 'distiluse-base-multilingual-cased-v2'
st_model = SentenceTransformer(checkpoint)

def generate_embeddings(text: str) -> list[float]:
    return st_model.encode(text or '').tolist()

Columns: {'AppID': Value('int64'), 'Name': Value('string'), 'Release date': Value('string'), 'Estimated owners': Value('string'), 'Peak CCU': Value('int64'), 'Required age': Value('int64'), 'Price': Value('float64'), 'DLC count': Value('int64'), 'About the game': Value('string'), 'Supported languages': Value('string'), 'Full audio languages': Value('string'), 'Reviews': Value('string'), 'Header image': Value('string'), 'Website': Value('string'), 'Support url': Value('string'), 'Support email': Value('string'), 'Windows': Value('bool'), 'Mac': Value('bool'), 'Linux': Value('bool'), 'Metacritic score': Value('int64'), 'Metacritic url': Value('string'), 'User score': Value('int64'), 'Positive': Value('int64'), 'Negative': Value('int64'), 'Score rank': Value('float64'), 'Achievements': Value('int64'), 'Recommendations': Value('int64'), 'Notes': Value('string'), 'Average playtime forever': Value('int64'), 'Average playtime two weeks': Value('int64'), 'Median playtime forever': Value('int64

In [9]:
def insert_games(engine, dataset):
    with tqdm(total=len(dataset)) as pbar:
        for item in dataset:
            game_description = item.get('About the game') or ''

            if not game_description:
                pbar.update(1)
                continue

            emb = generate_embeddings(game_description)
            name = item.get('Name')
            windows = bool(item.get('Windows'))
            linux = bool(item.get('Linux'))
            mac = bool(item.get('Mac'))
            price = float(item.get('Price') or 0.0)

            if name and game_description:
                with Session(engine) as session:
                    session.add(Games(
                        name=name,
                        description=(game_description or '')[:4096],
                        windows=windows,
                        linux=linux,
                        mac=mac,
                        price=price,
                        game_description_embedding=emb,
                    ))
                    session.commit()
            pbar.update(1)

insert_games(engine, dataset)

100%|██████████| 4000/4000 [07:39<00:00,  8.70it/s]


In [ ]:
def find_game(engine, 
              game_description: str, 
              windows: Optional[bool] = None, 
              linux: Optional[bool] = None, 
              mac: Optional[bool] = None, 
              price: Optional[float] = None):
    game_embedding = generate_embeddings(game_description)
    with Session(engine) as session:
        query = select(Games).order_by(
            Games.game_description_embedding.cosine_distance(game_embedding))
        if price is not None:
            query = query.filter(Games.price <= price)
        if windows:
            query = query.filter(Games.windows)
        if linux:
            query = query.filter(Games.linux)
        if mac:
            query = query.filter(Games.mac)
        result = session.execute(query, execution_options={"prebuffer_rows": True})
        return result.scalars().first()

g = find_game(engine, 'This is a game about a hero who saves the world', price=10)
print(g.name if g else None)

g = find_game(engine, game_description='Home decorating', price=20)
print(g.name if g else None)

g = find_game(engine, game_description='Home decorating', mac=True, price=5)
print(g.name if g else None)

幻想小镇危机
Our Home
Thanksgiving Day Mosaic


Milvus RAG setup and query

In [11]:
# Connect to Milvus
host = 'localhost'
port = '19530'
milvus_client = MilvusClient(host=host, port=port)

VECTOR_LENGTH = 768  # Silver Retriever base v1.1 outputs 768-d vectors
id_field = FieldSchema(name='id', dtype=DataType.INT64, is_primary=True, description='Primary id')
text = FieldSchema(name='text', dtype=DataType.VARCHAR, max_length=4096, description='Page text')
embedding_text = FieldSchema('embedding', dtype=DataType.FLOAT_VECTOR, dim=VECTOR_LENGTH, description='Embedded text')
fields = [id_field, text, embedding_text]
schema = CollectionSchema(fields=fields, auto_id=True, enable_dynamic_field=True, description='RAG Texts collection')

COLLECTION_NAME = 'rag_texts_and_embeddings'
if COLLECTION_NAME not in milvus_client.list_collections():
    milvus_client.create_collection(collection_name=COLLECTION_NAME, schema=schema)
    index_params = milvus_client.prepare_index_params()
    index_params.add_index(field_name='embedding', index_type='HNSW', metric_type='L2', params={'M': 4, 'efConstruction': 64})
    milvus_client.create_index(collection_name=COLLECTION_NAME, index_params=index_params)

print(milvus_client.list_collections())
print(milvus_client.describe_collection(COLLECTION_NAME))

['rag_texts_and_embeddings']
{'collection_name': 'rag_texts_and_embeddings', 'auto_id': True, 'num_shards': 1, 'description': 'RAG Texts collection', 'fields': [{'field_id': 100, 'name': 'id', 'description': 'Primary id', 'type': <DataType.INT64: 5>, 'params': {}, 'auto_id': True, 'is_primary': True}, {'field_id': 101, 'name': 'text', 'description': 'Page text', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 4096}}, {'field_id': 102, 'name': 'embedding', 'description': 'Embedded text', 'type': <DataType.FLOAT_VECTOR: 101>, 'params': {'dim': 768}}], 'functions': [], 'aliases': [], 'collection_id': 461828322862301844, 'consistency_level': 2, 'properties': {}, 'num_partitions': 1, 'enable_dynamic_field': True, 'created_timestamp': 461828356988207110}


In [12]:
# Download and process PDF into pages and embeddings
data_dir = './data'
os.makedirs(data_dir, exist_ok=True)

pdf_url = 'https://www.iab.org.pl/wp-content/uploads/2024/04/Przewodnik-po-sztucznej-inteligencji-2024_IAB-Polska.pdf'
file_name = 'Przewodnik-po-sztucznej-inteligencji-2024_IAB-Polska.pdf'
file_json = 'Przewodnik-po-sztucznej-inteligencji-2024_IAB-Polska.json'
embeddings_json = 'Przewodnik-po-sztucznej-inteligencji-2024_IAB-Polska-Embeddings.json'

def download_pdf_data(pdf_url: str, file_name: str) -> None:
    dest = os.path.join(data_dir, file_name)

    if os.path.exists(dest):
        return

    r = requests.get(pdf_url, stream=True)
    r.raise_for_status()

    with open(dest, 'wb') as f:
        for block in r.iter_content(1024):
            if block:
                f.write(block)

def extract_pdf_text(file_name, file_json):
    document = fitz.open(os.path.join(data_dir, file_name))
    pages = []

    for page_num in range(len(document)):
        page = document.load_page(page_num)
        page_text = page.get_text()
        pages.append({'page_num': page_num, 'text': page_text})

    with open(os.path.join(data_dir, file_json), 'w', encoding='utf-8') as f:
        json.dump(pages, f, indent=2, ensure_ascii=False)

def generate_embeddings(file_json, embeddings_json, model):
    with open(os.path.join(data_dir, file_json), 'r', encoding='utf-8') as f:
        data = json.load(f)

    pages = [p['text'] for p in data]
    embeddings = model.encode(pages)
    out = [{'page_num': i, 'embedding': embeddings[i].tolist()} for i in range(len(embeddings))]

    with open(os.path.join(data_dir, embeddings_json), 'w', encoding='utf-8') as f:
        json.dump(out, f, indent=2, ensure_ascii=False)

download_pdf_data(pdf_url, file_name)
extract_pdf_text(file_name, file_json)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
retriever_model = SentenceTransformer('ipipan/silver-retriever-base-v1.1', device=device)
generate_embeddings(file_json, embeddings_json, retriever_model)

In [13]:
# Insert into Milvus and search
with open(os.path.join(data_dir, file_json), 'r', encoding='utf-8') as t_f, open(os.path.join(data_dir, embeddings_json), 'r', encoding='utf-8') as e_f:
    text_data, embedding_data = json.load(t_f), json.load(e_f)

rows = [{"text": t['text'], "embedding": e['embedding']} for t, e in zip(text_data, embedding_data)]
milvus_client.insert(collection_name='rag_texts_and_embeddings', data=rows)
milvus_client.load_collection('rag_texts_and_embeddings')

def search(model, query, client=milvus_client):
    embedded_query = model.encode(query).tolist()
    result = client.search(collection_name='rag_texts_and_embeddings', data=[embedded_query], limit=3, search_params={'metric_type': 'L2'}, output_fields=['text'])
    return result

sr = search(retriever_model, 'Czym jest sztuczna inteligencja')
sr

data: [[{'id': 461828322862302640, 'distance': 29.125165939331055, 'entity': {'text': 'Historia powstania\nsztucznej inteligencji\n7\nW języku potocznym „sztuczny" oznacza to, co\njest \nwytworem \nmającym \nnaśladować \ncoś\nnaturalnego. W takim znaczeniu używamy\nterminu ,,sztuczny\'\', gdy mówimy o sztucznym\nlodowisku lub oku. Sztuczna inteligencja byłaby\nczymś (programem, maszyną) symulującym\ninteligencję naturalną, ludzką.\nSztuczna inteligencja (AI) to obszar informatyki,\nktóry skupia się na tworzeniu programów\nkomputerowych zdolnych do wykonywania\nzadań, które wymagają ludzkiej inteligencji. \nTe zadania obejmują rozpoznawanie wzorców,\nrozumienie języka naturalnego, podejmowanie\ndecyzji, uczenie się, planowanie i wiele innych.\nGłównym celem AI jest stworzenie systemów,\nktóre są zdolne do myślenia i podejmowania\ndecyzji na sposób przypominający ludzki.\nHistoria sztucznej inteligencji sięga lat 50. \nXX wieku, kiedy to powstały pierwsze koncepcje\ni modele tego, co mog

In [ ]:
if "GEMINI_KEY" not in globals():
    GEMINI_KEY = os.getenv('GEMINI_API_KEY')

In [16]:
gemini_client = genai.Client(api_key=GEMINI_KEY)
MODEL = 'gemini-2.0-flash'

def generate_response(prompt: str):
    try:
        response = gemini_client.models.generate_content(model=MODEL, contents=prompt)
        return response.text
    except Exception as e:
        print(f'Error generating response: {e}')
        return None

In [18]:
# Build RAG prompt and answer
def build_prompt(context: str, query: str) -> str:
    return (
        "Odpowiedz krotko i rzeczowo na pytanie uzytkownika po polsku. "
        "Uzyj wylacznie ponizszego kontekstu. Jesli kontekst nie zawiera odpowiedzi, powiedz, ze nie wiesz.\n"
        f"\nKontekst:\n{context}\n"
        f"\nPytanie: {query}\nOdpowiedz:"
    )


def rag(model, query: str) -> str:
    hits = search(model, query)
    # Extract top texts from Milvus hits
    top_texts: list[str] = []
    if hits and len(hits) > 0:
        for hit in hits[0]:
            ent = hit.get("entity")
            if isinstance(ent, dict):
                top_texts.append(ent.get("text", ""))
            else:
                top_texts.append(hit.get("text", ""))
    context = "\n---\n".join([t for t in top_texts if t])
    prompt = build_prompt(context, query)
    return generate_response(prompt) # type: ignore

rag(retriever_model, "Jakie sa glowne zastosowania sztucznej inteligencji?")

'Sztuczna inteligencja generatywna ma potencjał zastosowania w wielu dziedzinach, takich jak: sztuka, projektowanie, produkcja treści multimedialnych, czy nawet medycyna.\n'